In [3]:
# imports
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
import pandas as pd
from backtesting import Backtest
from ta.volatility import AverageTrueRange
from ta.momentum import RSIIndicator

from datetime import timedelta

In [4]:
# kaufman_efficiency_ratio ER
import pandas as pd
import numpy as np

def kaufman_efficiency_ratio(close_series, period=10):
    # Variation nette sur la période (entre maintenant et il y a N périodes)
    net_change = np.abs(close_series - close_series.shift(period))
    
    # Somme des variations absolues bar à bar sur la période
    change = np.abs(close_series - close_series.shift(1))
    volatility = change.rolling(window=period).sum()
    
    # Éviter la division par zéro
    er = np.where(volatility == 0, 0, net_change / volatility)
    
    return pd.Series(er, index=close_series.index)

# Exemple d'utilisation :
# df['KER'] = kaufman_efficiency_ratio(df['Close'], period=10)

In [5]:
# swing structure
def implement_market_structure_states_strict(df, n=5):
    """Calcule les états du marché avec transition obligatoire par l'état NEUTRAL.

    Aucun passage direct entre BULLISH et BEARISH n'est autorisé.
    """
    df = df.copy()
    length = len(df)

    df['is_swing_high'] = False
    df['is_swing_low'] = False
    for k in range(n, len(df) - n):
        # Le plus haut des n bougies précédentes et suivantes
        high_range = df['High'].iloc[k - n:k + n + 1]
        if df['High'].iloc[k] == high_range.max():
            df.loc[df.index[k], 'is_swing_high'] = True
            
        # Le plus bas des n bougies précédentes et suivantes
        low_range = df['Low'].iloc[k - n:k + n + 1]
        if df['Low'].iloc[k] == low_range.min():
            df.loc[df.index[k], 'is_swing_low'] = True

    states = []
    current_state = 'NEUTRAL'

    # Variables de structure de marché
    last_confirmed_high = None
    last_confirmed_low = None
    higher_low = None  # Le plancher (uniquement actif en BULLISH)
    lower_high = None  # Le plafond (uniquement actif en BEARISH)

    for j in range(length):
        close_p = df["Close"].iloc[j]

        # --- ÉTAPE B : Machine à états avec transit strict par NEUTRAL ---
        if current_state == "BULLISH":
            # Sortie de Bullish : Clôture sous le Higher Low (HL) -> NEUTRAL uniquement
            if higher_low is not None and close_p < higher_low:
                current_state = "NEUTRAL"
                higher_low = None  # Reset des repères obsolètes
                lower_high = None

        elif current_state == "BEARISH":
            # Sortie de Bearish : Clôture au-dessus du Lower High (LH) -> NEUTRAL uniquement
            if lower_high is not None and close_p > lower_high:
                current_state = "NEUTRAL"
                higher_low = None
                lower_high = None  # Reset des repères obsolètes

        elif current_state == "NEUTRAL":
            # En zone neutre, on attend un NOUVEAU BOS pour définir la tendance
            # Entrée en Bullish : Clôture au-dessus du dernier sommet confirmé
            # il faut tjs un high and un low avant de former une structure bearish ou bullish
            if last_confirmed_high is not None and last_confirmed_low is not None and close_p > last_confirmed_high:
                current_state = "BULLISH"
                higher_low = (
                    last_confirmed_low  # On fige le dernier creux comme HL
                )

            # Entrée en Bearish : Clôture sous le dernier creux confirmé
            elif last_confirmed_high is not None and last_confirmed_low is not None and close_p < last_confirmed_low:
                current_state = "BEARISH"
                lower_high = (
                    last_confirmed_high  # On fige le dernier sommet comme LH
                )

        states.append(current_state)

         # --- ÉTAPE A : Enregistrement des Swings dès qu'ils se confirment ---
        if df["is_swing_high"].iloc[j] == True:
            last_confirmed_high = df["High"].iloc[j]
            if current_state == "BEARISH" and lower_high > last_confirmed_high:
                lower_high = last_confirmed_high

        if df["is_swing_low"].iloc[j] == True:
            last_confirmed_low = df["Low"].iloc[j]
            if current_state == "BULLISH" and higher_low < last_confirmed_low:
                higher_low = last_confirmed_low

    df["market_state"] = states
    return df


In [6]:
# range box

def is_range_box(highs, lows, closes, N=20,
                  touch_tol_pct=0.10, min_touches=2,
                  overlap_threshold=0.60, position_threshold=0.25):
    """
    highs, lows, closes: mảng giá Daily ĐÃ ĐÓNG, phần tử cuối cùng là nến gần nhất
                          (không bao gồm nến Daily đang chạy dở, tránh lookahead).
    Trả về: (is_range: bool, details: dict)
    """
    h = np.array(highs[-N:])
    l = np.array(lows[-N:])
    c = closes.iloc[-1]

    box_high = h.max()
    box_low = l.min()
    box_height = box_high - box_low
    if box_height <= 0:
        return False, {}

    # 1. Overlap giữa các nến liên tiếp
    overlaps = 0
    for i in range(1, N):
        if l[i] <= h[i-1] and h[i] >= l[i-1]:
            overlaps += 1
    overlap_ratio = overlaps / (N - 1)

    # 2. Số lần chạm biên trên / dưới
    tol = box_height * touch_tol_pct
    upper_touches = int(np.sum(h >= box_high - tol))
    lower_touches = int(np.sum(l <= box_low + tol))

    # 3. Vị trí close hiện tại so với trung điểm box
    mid = (box_high + box_low) / 2
    position_ratio = abs(c - mid) / box_height

    is_range = (
        overlap_ratio >= overlap_threshold
        and upper_touches >= min_touches
        and lower_touches >= min_touches
        and position_ratio <= position_threshold
    )

    details = {
        "box_high": box_high, "box_low": box_low, "box_height": box_height,
        "overlap_ratio": round(overlap_ratio, 3),
        "upper_touches": upper_touches, "lower_touches": lower_touches,
        "position_ratio": round(position_ratio, 3),
    }
    return is_range, details

In [7]:
# rsi trending stratey
from backtesting import Strategy

global_tp_multiple_rsi = 0
global_stop_trade_rsi = 0
global_start_trade_hour = 0
global_er15_threshold = 0

class OpeningRangeBreakout(Strategy):
    # variables
    tp_multiple_rsi = 5
    start_trade_hour = 5
    stop_trade_rsi = 16
    er15_threshold = 0.3

    # fixed variables
    risk_percent = 0.01
    max_leverage = 4
    my_equity = 10000

    # def init(self, range_to_consider, tp_multiple, stop_trade_hour, close_range_hour):
    def init(self):
        self.current_day = None
        self.current_day_order_placed = False
        self.last_low = None
        self.last_high = None
        self.order_placed_hour = None
        self.order_executed = False

        if global_tp_multiple_rsi != 0:
            self.tp_multiple_rsi = global_tp_multiple_rsi

        if global_stop_trade_rsi != 0:
            self.stop_trade_rsi = global_stop_trade_rsi

        if global_start_trade_hour != 0:
            self.start_trade_hour = global_start_trade_hour

        if global_er15_threshold != 0:
            self.er15_threshold = global_er15_threshold

    def reset_range(self, day):
        # delete placed order at the end of the day if not executed
        if self.current_day_order_placed is True and self.order_placed_hour:
            for trade in self.trades:
                if trade.tag == self.get_tag():
                    self.order_executed = True

            if self.order_executed == False:
                for order in self.orders:
                    if order.tag == self.get_tag():
                        order.cancel()

        self.current_day = day
        self.current_day_order_placed = False
        self.order_placed_hour = None
        self.last_low = None
        self.last_high = None
        self.order_executed = False

    def get_tag(self):
        return f"{self.current_day} {self.order_placed_hour}"

    def place_long_order(self, candle_date):
        planned_entry_price = self.data.High[-1]
        sl_price = self.last_low
        per_share_risk = abs(planned_entry_price - sl_price)
        tp_multiple_rsi = self.tp_multiple_rsi 
        tp_price = planned_entry_price + per_share_risk * tp_multiple_rsi
        position_size = int((self.risk_percent * self.equity) / per_share_risk)

        # print(candle_date)
        # print(f"long per_share_risk = {per_share_risk} & atr = {self.data['ATR'][-1]}")
        if per_share_risk < self.data['ATR'][-1] * 2.5 and per_share_risk > self.data['ATR'][-1] * 1:
            # print(f'Go long limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')
            self.order_placed_hour = candle_date.time().hour
            self.buy(size = position_size, stop=planned_entry_price, tp=tp_price, sl=sl_price, tag=self.get_tag())
            self.current_day_order_placed = True
        
    def place_short_order(self, candle_date):
        planned_entry_price = self.data.Low[-1]
        sl_price = self.last_high
        per_share_risk = abs(planned_entry_price - sl_price)
        tp_multiple_rsi = self.tp_multiple_rsi 
        tp_price = planned_entry_price - per_share_risk * tp_multiple_rsi
        position_size = int((self.risk_percent * self.equity) / per_share_risk)

        # print(candle_date)
        # print(f"short per_share_risk = {per_share_risk} & atr = {self.data['ATR'][-1]}")
        if per_share_risk < self.data['ATR'][-1] * 2.5 and per_share_risk > self.data['ATR'][-1] * 1:
            # print(f'Go short limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')
            self.order_placed_hour = candle_date.time().hour
            self.sell(size = position_size, stop=planned_entry_price, tp=tp_price, sl=sl_price, tag=self.get_tag())
            self.current_day_order_placed = True
            
    def use_rsi_entry_strategy_market_ranging(self, candle_datetime):
        c_long = self.data['RSI14'][-1] < 35 and self.data['market_state_h4'][-1] == 'BULLISH'
        c_short = self.data['RSI14'][-1] > 65 and self.data['market_state_h4'][-1] == 'BEARISH'

        if not self.current_day_order_placed:
            # print(f"not placed RSI14 = {self.data['RSI14'][-1]}")
            if c_long:
                self.place_long_order(candle_datetime)

            if c_short:
                self.place_short_order(candle_datetime)
        else:
            for trade in self.trades:
                if trade.tag == self.get_tag():
                    self.order_executed = True

            # print(f"placed RSI14 = {self.data['RSI14'][-1]}")
            if self.order_executed == False:
                for order in self.orders:
                    if order.tag == self.get_tag():
                        order.cancel()
                        self.current_day_order_placed = False
                        if order.is_long:
                            self.place_long_order(candle_datetime)
                        else:
                            self.place_short_order(candle_datetime)

    def next(self):
        candle_datetime = self.data.index[-1]
        current_bar_date = self.data.index[-1].date()
        if self.current_day != current_bar_date or candle_datetime.time().hour >= self.stop_trade_rsi:
            self.reset_range(current_bar_date)

        if self.last_low is None:
            self.last_low = self.data.Low[-1]
        else:
            if self.last_low > self.data.Low[-1]:
                self.last_low = self.data.Low[-1]

        if self.last_high is None:
            self.last_high = self.data.High[-1]
        else:
            if self.last_high < self.data.High[-1]:
                self.last_high = self.data.High[-1]

        c_range_box = self.data['is_range_box'][-1] == False
        c_er = self.data['ER'][-1] > self.er15_threshold
        c_time_limit = candle_datetime.time().hour < self.stop_trade_rsi and candle_datetime.time().hour > self.start_trade_hour
        
        if c_time_limit:
            # print(candle_datetime)
            # print(f"c_market_structure = {self.data['market_state'][-1]} & c_er = {self.data['ER'][-1]}")
            self.use_rsi_entry_strategy_market_ranging(candle_datetime)

        # if self.current_day_order_placed == True:
        #     print(candle_datetime)
        #     print(self.orders)
        #     print(self.trades)
        # print("--------------------------------")
            



In [8]:
# rsi ranging stratey
from backtesting import Strategy

global_tp_multiple_rsi = 0
global_stop_trade_rsi = 0
global_start_trade_hour = 0
global_er15_threshold = 0

class OpeningRangeBreakout(Strategy):
    # variables
    tp_multiple_rsi = 5
    start_trade_hour = 5
    stop_trade_rsi = 16
    er15_threshold = 0.3

    # fixed variables
    risk_percent = 0.01
    max_leverage = 4
    my_equity = 10000

    # def init(self, range_to_consider, tp_multiple, stop_trade_hour, close_range_hour):
    def init(self):
        self.current_day = None
        self.current_day_order_placed = False
        self.last_low = None
        self.last_high = None
        self.order_placed_hour = None
        self.order_executed = False

        if global_tp_multiple_rsi != 0:
            self.tp_multiple_rsi = global_tp_multiple_rsi

        if global_stop_trade_rsi != 0:
            self.stop_trade_rsi = global_stop_trade_rsi

        if global_start_trade_hour != 0:
            self.start_trade_hour = global_start_trade_hour

        if global_er15_threshold != 0:
            self.er15_threshold = global_er15_threshold

    def reset_range(self, day):
        # delete placed order at the end of the day if not executed
        if self.current_day_order_placed is True and self.order_placed_hour:
            for trade in self.trades:
                if trade.tag == self.get_tag():
                    self.order_executed = True

            if self.order_executed == False:
                for order in self.orders:
                    if order.tag == self.get_tag():
                        order.cancel()

        self.current_day = day
        self.current_day_order_placed = False
        self.order_placed_hour = None
        self.last_low = None
        self.last_high = None
        self.order_executed = False

    def get_tag(self):
        return f"{self.current_day} {self.order_placed_hour}"

    def place_long_order(self, candle_date):
        planned_entry_price = self.data.High[-1]
        sl_price = self.last_low
        per_share_risk = abs(planned_entry_price - sl_price)
        tp_multiple_rsi = self.tp_multiple_rsi 
        tp_price = planned_entry_price + per_share_risk * tp_multiple_rsi
        position_size = int((self.risk_percent * self.equity) / per_share_risk)

        # print(candle_date)
        # print(f"long per_share_risk = {per_share_risk} & atr = {self.data['ATR'][-1]}")
        if per_share_risk < self.data['ATR'][-1] * 2.5 and per_share_risk > self.data['ATR'][-1] * 1:
            # print(f'Go long limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')
            self.order_placed_hour = candle_date.time().hour
            self.buy(size = position_size, stop=planned_entry_price, tp=tp_price, sl=sl_price, tag=self.get_tag())
            self.current_day_order_placed = True
        
    def place_short_order(self, candle_date):
        planned_entry_price = self.data.Low[-1]
        sl_price = self.last_high
        per_share_risk = abs(planned_entry_price - sl_price)
        tp_multiple_rsi = self.tp_multiple_rsi 
        tp_price = planned_entry_price - per_share_risk * tp_multiple_rsi
        position_size = int((self.risk_percent * self.equity) / per_share_risk)

        # print(candle_date)
        # print(f"short per_share_risk = {per_share_risk} & atr = {self.data['ATR'][-1]}")
        if per_share_risk < self.data['ATR'][-1] * 2.5 and per_share_risk > self.data['ATR'][-1] * 1:
            # print(f'Go short limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')
            self.order_placed_hour = candle_date.time().hour
            self.sell(size = position_size, stop=planned_entry_price, tp=tp_price, sl=sl_price, tag=self.get_tag())
            self.current_day_order_placed = True
            
    def use_rsi_entry_strategy_market_ranging(self, candle_datetime):
        c_long = self.data['RSI14'][-1] < 35
        c_short = self.data['RSI14'][-1] > 65

        if not self.current_day_order_placed:
            # print(f"not placed RSI14 = {self.data['RSI14'][-1]}")
            if c_long:
                self.place_long_order(candle_datetime)

            if c_short:
                self.place_short_order(candle_datetime)
        else:
            for trade in self.trades:
                if trade.tag == self.get_tag():
                    self.order_executed = True

            # print(f"placed RSI14 = {self.data['RSI14'][-1]}")
            if self.order_executed == False:
                for order in self.orders:
                    if order.tag == self.get_tag():
                        order.cancel()
                        self.current_day_order_placed = False
                        if order.is_long:
                            self.place_long_order(candle_datetime)
                        else:
                            self.place_short_order(candle_datetime)

    def next(self):
        candle_datetime = self.data.index[-1]
        current_bar_date = self.data.index[-1].date()
        if self.current_day != current_bar_date or candle_datetime.time().hour >= self.stop_trade_rsi:
            self.reset_range(current_bar_date)

        if self.last_low is None:
            self.last_low = self.data.Low[-1]
        else:
            if self.last_low > self.data.Low[-1]:
                self.last_low = self.data.Low[-1]

        if self.last_high is None:
            self.last_high = self.data.High[-1]
        else:
            if self.last_high < self.data.High[-1]:
                self.last_high = self.data.High[-1]

        c_market_structure = self.data['market_state'][-1] == 'NEUTRAL'
        c_range_box = self.data['is_range_box'][-1] == True
        c_er = self.data['ER'][-1] < self.er15_threshold
        c_time_limit = candle_datetime.time().hour < self.stop_trade_rsi and candle_datetime.time().hour > self.start_trade_hour
        
        if c_time_limit and c_market_structure and c_range_box:
            # print(candle_datetime)
            # print(f"c_market_structure = {self.data['market_state'][-1]} & c_er = {self.data['ER'][-1]}")
            self.use_rsi_entry_strategy_market_ranging(candle_datetime)

        # if self.current_day_order_placed == True:
        #     print(candle_datetime)
        #     print(self.orders)
        #     print(self.trades)
        # print("--------------------------------")
            



In [9]:
# rsi trending stratey + m15 structure
from backtesting import Strategy

global_tp_multiple_rsi = 0
global_stop_trade_rsi = 0
global_start_trade_hour = 0
global_er15_threshold = 0

class OpeningRangeBreakout(Strategy):
    # variables
    tp_multiple_rsi = 5
    start_trade_hour = 5
    stop_trade_rsi = 16
    er15_threshold = 0.3

    # fixed variables
    risk_percent = 0.01
    max_leverage = 4
    my_equity = 10000

    # def init(self, range_to_consider, tp_multiple, stop_trade_hour, close_range_hour):
    def init(self):
        self.current_day = None
        self.current_day_order_placed = False
        self.last_low = None
        self.last_high = None
        self.order_placed_hour = None
        self.order_executed = False
        self.is_rsi_touched = False

        # when rsi is touched then we want trend to switch to bullish/bearish
        self.transition_state = None

        if global_tp_multiple_rsi != 0:
            self.tp_multiple_rsi = global_tp_multiple_rsi

        if global_stop_trade_rsi != 0:
            self.stop_trade_rsi = global_stop_trade_rsi

        if global_start_trade_hour != 0:
            self.start_trade_hour = global_start_trade_hour

        if global_er15_threshold != 0:
            self.er15_threshold = global_er15_threshold

    def reset_range(self, day):
        # delete placed order at the end of the day if not executed
        if self.current_day_order_placed is True and self.order_placed_hour:
            for trade in self.trades:
                if trade.tag == self.get_tag():
                    self.order_executed = True

            if self.order_executed == False:
                for order in self.orders:
                    if order.tag == self.get_tag():
                        order.cancel()

        self.current_day = day
        self.current_day_order_placed = False
        self.order_placed_hour = None
        self.last_low = None
        self.last_high = None
        self.order_executed = False
        self.is_rsi_touched = False
        self.transition_state = None

    def get_tag(self):
        return f"{self.current_day} {self.order_placed_hour}"

    def place_long_order(self, candle_date):
        if self.last_high is not None:
            planned_entry_price = self.last_high
            sl_price = self.last_low
            per_share_risk = abs(planned_entry_price - sl_price)
            tp_multiple_rsi = self.tp_multiple_rsi 
            tp_price = planned_entry_price + per_share_risk * tp_multiple_rsi
            position_size = int((self.risk_percent * self.equity) / per_share_risk)

            # print(candle_date)
            # print(f'limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')

            # print(f"long per_share_risk = {per_share_risk} & atr = {self.data['ATR'][-1]}")
            # if per_share_risk < self.data['ATR'][-1] * 5 and per_share_risk > self.data['ATR'][-1] * 1:
            if planned_entry_price > sl_price:
                # print(f'Go long limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')
                self.order_placed_hour = candle_date.time().hour
                self.buy(size = position_size, limit=planned_entry_price, tp=tp_price, sl=sl_price, tag=self.get_tag())
                self.current_day_order_placed = True
        
    def place_short_order(self, candle_date):
        if self.last_low is not None:
            planned_entry_price = self.last_low
            sl_price = self.last_high
            per_share_risk = abs(planned_entry_price - sl_price)
            tp_multiple_rsi = self.tp_multiple_rsi 
            tp_price = planned_entry_price - per_share_risk * tp_multiple_rsi
            position_size = int((self.risk_percent * self.equity) / per_share_risk)

            # print(candle_date)
            # print(f'limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')

            # print(f"short per_share_risk = {per_share_risk} & atr = {self.data['ATR'][-1]}")
            # if per_share_risk < self.data['ATR'][-1] * 5 and per_share_risk > self.data['ATR'][-1] * 1:
            if planned_entry_price < sl_price:
                # print(f'Go short limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')
                self.order_placed_hour = candle_date.time().hour
                self.sell(size = position_size, limit=planned_entry_price, tp=tp_price, sl=sl_price, tag=self.get_tag())
                self.current_day_order_placed = True
            
    def use_rsi_entry_strategy_market_ranging(self, candle_datetime):
        if self.is_rsi_touched == False:
            self.is_rsi_touched = self.data['RSI14'][-1] < 35 and self.data['market_state_h4'][-1] == 'BULLISH' or self.data['RSI14'][-1] > 65 and self.data['market_state_h4'][-1] == 'BEARISH'

        if self.is_rsi_touched is not None:
            if self.data['RSI14'][-1] > 65 and self.data['market_state_h4'][-1] == 'BULLISH' or self.data['RSI14'][-1] < 35 and self.data['market_state_h4'][-1] == 'BEARISH':
                self.is_rsi_touched = False
            
        c_long = self.data['market_state_h4'][-1] == 'BULLISH' and self.data['market_state'][-1] == 'BULLISH'
        c_short = self.data['market_state_h4'][-1] == 'BEARISH' and self.data['market_state'][-1] == 'BEARISH'

        # when rsi is touched then we want to see market state change from NEUTRAL to BULLISH/BEARISH
        c_transition = self.transition_state == 'NEUTRAL'

        if not self.current_day_order_placed and self.is_rsi_touched:
            # print(f"not placed RSI14 = {self.data['RSI14'][-1]}")
            if self.data['market_state'][-1] == 'NEUTRAL':
                self.transition_state = 'NEUTRAL'

            if c_long and c_transition:
                self.place_long_order(candle_datetime)

            if c_short and c_transition:
                self.place_short_order(candle_datetime)
        else:
            for trade in self.trades:
                if trade.tag == self.get_tag():
                    self.order_executed = True

            # print(f"placed RSI14 = {self.data['RSI14'][-1]}")
            # if self.order_executed == False:
            #     for order in self.orders:
            #         if order.tag == self.get_tag():
            #             order.cancel()
            #             self.current_day_order_placed = False
            #             if order.is_long:
            #                 self.place_long_order(candle_datetime)
            #             else:
            #                 self.place_short_order(candle_datetime)

    def next(self):
        candle_datetime = self.data.index[-1]
        current_bar_date = self.data.index[-1].date()
        if self.current_day != current_bar_date or candle_datetime.time().hour >= self.stop_trade_rsi:
            self.reset_range(current_bar_date)

        # print(self.data['is_swing_low'].shift(2))
        if self.data['is_swing_low'][len(self.data['is_swing_low'])-3] == True:
            self.last_low = self.data.Low[len(self.data['is_swing_low'])-3]

        if self.data['is_swing_high'][len(self.data['is_swing_low'])-3] == True:
            self.last_high = self.data.High[len(self.data['is_swing_low'])-3]

        c_range_box = self.data['is_range_box'][-1] == False
        c_er = self.data['ER'][-1] > self.er15_threshold
        c_time_limit = candle_datetime.time().hour < self.stop_trade_rsi and candle_datetime.time().hour > self.start_trade_hour
        
        if c_time_limit:
            # print(candle_datetime)
            # print(f"c_market_structure = {self.data['market_state'][-1]} & c_er = {self.data['ER'][-1]}")
            self.use_rsi_entry_strategy_market_ranging(candle_datetime)

        # if self.current_day_order_placed == True:
        #     print(candle_datetime)
        #     print(self.orders)
        #     print(self.trades)
        # print("--------------------------------")
            



In [10]:
# rsi ranging stratey + m15 structure
from backtesting import Strategy

global_tp_multiple_rsi = 0
global_stop_trade_rsi = 0
global_start_trade_hour = 0
global_er15_threshold = 0

class OpeningRangeBreakout(Strategy):
    # variables
    tp_multiple_rsi = 5
    start_trade_hour = 5
    stop_trade_rsi = 16
    er15_threshold = 0.3

    # fixed variables
    risk_percent = 0.01
    max_leverage = 4
    my_equity = 10000

    # def init(self, range_to_consider, tp_multiple, stop_trade_hour, close_range_hour):
    def init(self):
        self.current_day = None
        self.current_day_order_placed = False
        self.last_low = None
        self.lower_low = None
        self.last_high = None
        self.higher_high = None
        self.order_placed_hour = None
        self.order_executed = False
        self.is_rsi_touched = False

        if global_tp_multiple_rsi != 0:
            self.tp_multiple_rsi = global_tp_multiple_rsi

        if global_stop_trade_rsi != 0:
            self.stop_trade_rsi = global_stop_trade_rsi

        if global_start_trade_hour != 0:
            self.start_trade_hour = global_start_trade_hour

        if global_er15_threshold != 0:
            self.er15_threshold = global_er15_threshold

    def reset_range(self, day):
        # delete placed order at the end of the day if not executed
        if self.current_day_order_placed is True and self.order_placed_hour:
            for trade in self.trades:
                if trade.tag == self.get_tag():
                    self.order_executed = True

            if self.order_executed == False:
                for order in self.orders:
                    if order.tag == self.get_tag():
                        order.cancel()

        self.current_day = day
        self.current_day_order_placed = False
        self.order_placed_hour = None
        self.last_low = None
        self.lower_low = None
        self.last_high = None
        self.higher_high = None
        self.order_executed = False
        self.is_rsi_touched = False

    def get_tag(self):
        return f"{self.current_day} {self.order_placed_hour}"

    def place_long_order(self, candle_date):
        if self.lower_low is not None and self.last_high is not None:
            planned_entry_price = self.last_high
            sl_price = self.lower_low
            per_share_risk = abs(planned_entry_price - sl_price)
            tp_multiple_rsi = self.tp_multiple_rsi 
            tp_price = planned_entry_price + per_share_risk * tp_multiple_rsi
            position_size = int((self.risk_percent * self.equity) / per_share_risk)

            print(candle_date)
            print(f'limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')

            # print(f"long per_share_risk = {per_share_risk} & atr = {self.data['ATR'][-1]}")
            # if per_share_risk < self.data['ATR'][-1] * 5 and per_share_risk > self.data['ATR'][-1] * 1:
            if planned_entry_price > sl_price:
                print(f'Go long limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')
                self.order_placed_hour = candle_date.time().hour
                self.buy(size = position_size, limit=planned_entry_price, tp=tp_price, sl=sl_price, tag=self.get_tag())
                self.current_day_order_placed = True
        
    def place_short_order(self, candle_date):
        if self.last_low is not None and self.higher_high is not None:
            planned_entry_price = self.last_low
            sl_price = self.higher_high
            per_share_risk = abs(planned_entry_price - sl_price)
            tp_multiple_rsi = self.tp_multiple_rsi 
            tp_price = planned_entry_price - per_share_risk * tp_multiple_rsi
            position_size = int((self.risk_percent * self.equity) / per_share_risk)

            print(candle_date)
            print(f'limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')

            # print(f"short per_share_risk = {per_share_risk} & atr = {self.data['ATR'][-1]}")
            # if per_share_risk < self.data['ATR'][-1] * 5 and per_share_risk > self.data['ATR'][-1] * 1:
            if planned_entry_price < sl_price:
                print(f'Go short limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')
                self.order_placed_hour = candle_date.time().hour
                self.sell(size = position_size, limit=planned_entry_price, tp=tp_price, sl=sl_price, tag=self.get_tag())
                self.current_day_order_placed = True
            
    def use_rsi_entry_strategy_market_ranging(self, candle_datetime):
        c_long = self.data['market_state_h4'][-1] == 'NEUTRAL' and self.data['market_state'][-1] == 'BULLISH' and self.data['RSI14'][-1] < 35
        c_short = self.data['market_state_h4'][-1] == 'NEUTRAL' and self.data['market_state'][-1] == 'BEARISH' and self.data['RSI14'][-1] > 65

        if not self.current_day_order_placed:
            # print(f"not placed RSI14 = {self.data['RSI14'][-1]}")
            if c_long:
                self.place_long_order(candle_datetime)

            if c_short:
                self.place_short_order(candle_datetime)
        else:
            for trade in self.trades:
                if trade.tag == self.get_tag():
                    self.order_executed = True

            # print(f"placed RSI14 = {self.data['RSI14'][-1]}")
            # if self.order_executed == False:
            #     for order in self.orders:
            #         if order.tag == self.get_tag():
            #             order.cancel()
            #             self.current_day_order_placed = False
            #             if order.is_long:
            #                 self.place_long_order(candle_datetime)
            #             else:
            #                 self.place_short_order(candle_datetime)

    def next(self):
        candle_datetime = self.data.index[-1]
        current_bar_date = self.data.index[-1].date()
        if self.current_day != current_bar_date or candle_datetime.time().hour >= self.stop_trade_rsi:
            self.reset_range(current_bar_date)

        if self.data['is_swing_low'][-1] == True:
            if self.lower_low is not None:
                self.last_low = self.lower_low 
            self.lower_low = self.data.Low[-1]

        if self.data['is_swing_high'][-1] == True:
            if self.higher_high is not None:
                self.last_high = self.higher_high
            self.higher_high = self.data.High[-1]

        c_range_box = self.data['is_range_box'][-1] == False
        c_er = self.data['ER'][-1] > self.er15_threshold
        c_time_limit = candle_datetime.time().hour < self.stop_trade_rsi and candle_datetime.time().hour > self.start_trade_hour
        
        if c_time_limit:
            # print(candle_datetime)
            # print(f"c_market_structure = {self.data['market_state'][-1]} & c_er = {self.data['ER'][-1]}")
            self.use_rsi_entry_strategy_market_ranging(candle_datetime)

        # if self.current_day_order_placed == True:
        #     print(candle_datetime)
        #     print(self.orders)
        #     print(self.trades)
        # print("--------------------------------")
            



In [43]:
# rsi trending stratey + m15 structure + ATR
from backtesting import Strategy

global_tp_multiple_rsi = 0
global_stop_trade_rsi = 0
global_start_trade_hour = 0
global_er15_threshold = 0

class OpeningRangeBreakout(Strategy):
    # variables
    tp_multiple_rsi = 5
    start_trade_hour = 5
    stop_trade_rsi = 16
    er15_threshold = 0.3

    # fixed variables
    risk_percent = 0.01
    max_leverage = 4
    my_equity = 10000

    # def init(self, range_to_consider, tp_multiple, stop_trade_hour, close_range_hour):
    def init(self):
        self.current_day = None
        self.current_day_order_placed = False
        self.last_low = None
        self.last_high = None
        self.order_placed_hour = None
        self.order_executed = False
        self.is_rsi_touched = False

        # when rsi is touched then we want trend to switch to bullish/bearish
        self.transition_state = None

        if global_tp_multiple_rsi != 0:
            self.tp_multiple_rsi = global_tp_multiple_rsi

        if global_stop_trade_rsi != 0:
            self.stop_trade_rsi = global_stop_trade_rsi

        if global_start_trade_hour != 0:
            self.start_trade_hour = global_start_trade_hour

        if global_er15_threshold != 0:
            self.er15_threshold = global_er15_threshold

    def reset_range(self, day):
        # delete placed order at the end of the day if not executed
        if self.current_day_order_placed is True and self.order_placed_hour:
            for trade in self.trades:
                if trade.tag == self.get_tag():
                    self.order_executed = True

            if self.order_executed == False:
                for order in self.orders:
                    if order.tag == self.get_tag():
                        order.cancel()

        self.current_day = day
        self.current_day_order_placed = False
        self.order_placed_hour = None
        self.last_low = None
        self.last_high = None
        self.order_executed = False
        self.is_rsi_touched = False
        self.transition_state = None

    def get_tag(self):
        return f"{self.current_day} {self.order_placed_hour}"

    def place_long_order(self, candle_date):
        if self.last_high is not None:
            planned_entry_price = self.last_high
            sl_price = planned_entry_price - self.data['ATR'][-1]
            per_share_risk = abs(planned_entry_price - sl_price)
            tp_multiple_rsi = self.tp_multiple_rsi 
            tp_price = planned_entry_price + per_share_risk * tp_multiple_rsi
            position_size = int((self.risk_percent * self.equity) / per_share_risk)

            # print(candle_date)
            # print(f'limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')

            # print(f"long per_share_risk = {per_share_risk} & atr = {self.data['ATR'][-1]}")
            # if per_share_risk < self.data['ATR'][-1] * 5 and per_share_risk > self.data['ATR'][-1] * 1:
            if planned_entry_price > sl_price:
                # print(f'Go long limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')
                self.order_placed_hour = candle_date.time().hour
                self.buy(size = position_size, limit=planned_entry_price, tp=tp_price, sl=sl_price, tag=self.get_tag())
                self.current_day_order_placed = True
        
    def place_short_order(self, candle_date):
        if self.last_low is not None:
            planned_entry_price = self.last_low
            sl_price = planned_entry_price + self.data['ATR'][-1]
            per_share_risk = abs(planned_entry_price - sl_price)
            tp_multiple_rsi = self.tp_multiple_rsi 
            tp_price = planned_entry_price - per_share_risk * tp_multiple_rsi
            position_size = int((self.risk_percent * self.equity) / per_share_risk)

            # print(candle_date)
            # print(f'limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')

            # print(f"short per_share_risk = {per_share_risk} & atr = {self.data['ATR'][-1]}")
            # if per_share_risk < self.data['ATR'][-1] * 5 and per_share_risk > self.data['ATR'][-1] * 1:
            if planned_entry_price < sl_price:
                # print(f'Go short limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')
                self.order_placed_hour = candle_date.time().hour
                self.sell(size = position_size, limit=planned_entry_price, tp=tp_price, sl=sl_price, tag=self.get_tag())
                self.current_day_order_placed = True
            
    def use_rsi_entry_strategy_market_ranging(self, candle_datetime):
        if self.is_rsi_touched == False:
            self.is_rsi_touched = self.data['RSI14'][-1] < 35 and self.data['market_state_h4'][-1] == 'BULLISH' or self.data['RSI14'][-1] > 65 and self.data['market_state_h4'][-1] == 'BEARISH'

        if self.is_rsi_touched is not None:
            if self.data['RSI14'][-1] > 65 and self.data['market_state_h4'][-1] == 'BULLISH' or self.data['RSI14'][-1] < 35 and self.data['market_state_h4'][-1] == 'BEARISH':
                self.is_rsi_touched = False
            
        c_long = self.data['market_state_h4'][-1] == 'BULLISH' and self.data['market_state'][-1] == 'BULLISH'
        c_short = self.data['market_state_h4'][-1] == 'BEARISH' and self.data['market_state'][-1] == 'BEARISH'

        # when rsi is touched then we want to see market state change from NEUTRAL to BULLISH/BEARISH
        c_transition = self.transition_state == 'NEUTRAL'

        if not self.current_day_order_placed and self.is_rsi_touched:
            # print(f"not placed RSI14 = {self.data['RSI14'][-1]}")
            if self.data['market_state'][-1] == 'NEUTRAL':
                self.transition_state = 'NEUTRAL'

            if c_long and c_transition:
                self.place_long_order(candle_datetime)

            if c_short and c_transition:
                self.place_short_order(candle_datetime)
        else:
            for trade in self.trades:
                if trade.tag == self.get_tag():
                    self.order_executed = True

            # print(f"placed RSI14 = {self.data['RSI14'][-1]}")
            # if self.order_executed == False:
            #     for order in self.orders:
            #         if order.tag == self.get_tag():
            #             order.cancel()
            #             self.current_day_order_placed = False
            #             if order.is_long:
            #                 self.place_long_order(candle_datetime)
            #             else:
            #                 self.place_short_order(candle_datetime)

    def next(self):
        candle_datetime = self.data.index[-1]
        current_bar_date = self.data.index[-1].date()
        if self.current_day != current_bar_date or candle_datetime.time().hour >= self.stop_trade_rsi:
            self.reset_range(current_bar_date)

        # print(self.data['is_swing_low'].shift(2))
        if self.data['is_swing_low'][len(self.data['is_swing_low'])-3] == True:
            self.last_low = self.data.Low[len(self.data['is_swing_low'])-3]

        if self.data['is_swing_high'][len(self.data['is_swing_low'])-3] == True:
            self.last_high = self.data.High[len(self.data['is_swing_low'])-3]

        c_range_box = self.data['is_range_box'][-1] == False
        c_er = self.data['ER'][-1] > self.er15_threshold
        c_time_limit = candle_datetime.time().hour < self.stop_trade_rsi and candle_datetime.time().hour > self.start_trade_hour
        
        if c_time_limit:
            # print(candle_datetime)
            # print(f"c_market_structure = {self.data['market_state'][-1]} & c_er = {self.data['ER'][-1]}")
            self.use_rsi_entry_strategy_market_ranging(candle_datetime)

        # if self.current_day_order_placed == True:
        #     print(candle_datetime)
        #     print(self.orders)
        #     print(self.trades)
        # print("--------------------------------")
            



In [63]:
# variables
# all strategy
window = 2
global_er15_threshold = 0.3

# rsi
global_start_trade_hour = 7
global_tp_multiple_rsi = 1
global_stop_trade_rsi = 19

# orb
global_range_to_consider = 3
global_close_range_hour = 7
global_tp_multiple_orb = 5
global_stop_trade_orb = 18

# start_date = '2023-1-1 00:00'
# end_date = '2024-3-29 00:00'
# start_date = '2025-1-1 00:00'
# end_date = '2025-12-30 00:00'
start_date = '2022-01-1 00:00'
end_date = '2026-9-16 00:00'
pair = 'GBPJPY'
timezone = "Europe/London"

# range box
window_rangebox = 15
touch_tol_pct = 0.15
min_touches = 3
overlap_threshold = 0.5
position_threshold = 0.2

In [55]:
# load data

df_h4_raw = pd.read_csv(f"./Trading Data/{pair}_H4.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_m15_raw = pd.read_csv(f"./Trading Data/{pair}_M15.csv",sep=',', parse_dates=['Date'], index_col='Date')

print(df_h4_raw)
print(df_m15_raw)

                        Open     High      Low    Close  Tickvol
Date                                                            
2010-09-10 04:00:00  129.605  129.630  128.995  129.385    25539
2010-09-10 08:00:00  129.380  129.985  129.275  129.785    44503
2010-09-10 12:00:00  129.785  130.065  129.425  129.570    54127
2010-09-10 16:00:00  129.580  129.770  129.120  129.135    39740
2010-09-10 20:00:00  129.130  129.335  129.120  129.255     8320
...                      ...      ...      ...      ...      ...
2026-09-23 20:00:00  209.580  209.701  209.431  209.564    16661
2026-09-24 00:00:00  209.566  209.638  208.768  209.072    51880
2026-09-24 04:00:00  209.072  209.859  209.019  209.838    38121
2026-09-24 08:00:00  209.840  210.067  209.417  209.996    62442
2026-09-24 12:00:00  209.997  210.105  209.846  209.848    47545

[25029 rows x 5 columns]
                        Open     High      Low    Close  Tickvol
Date                                                            

In [58]:
# process h4 data
df_h4 = df_h4_raw.loc[start_date : end_date]

df_range_box_list = [False for i in range(len(df_h4))]

for index in range(window_rangebox, len(df_h4)):
    start_index = index - window_rangebox
    range_ok, box_info = is_range_box(df_h4['High'][start_index:index], df_h4['Low'][start_index:index], df_h4['Close'][start_index:index], 
                                    N=window_rangebox, touch_tol_pct=touch_tol_pct, min_touches=min_touches, overlap_threshold=overlap_threshold, position_threshold=position_threshold)
                                
    df_range_box_list[index] = range_ok

df_h4['is_range_box'] = df_range_box_list

indicator_er = kaufman_efficiency_ratio(df_h4['Close'], 20)
df_result = implement_market_structure_states_strict(df_h4, n=window)
df_result['ER'] = indicator_er.values


df_h4_periode = df_result.loc['2026-06-10 00:00' : '2026-06-17 00:00']
# Affichage des transitions d'états
df_h4_periode["state_changed"] = df_h4_periode["market_state"] != df_h4_periode[
    "market_state"
].shift(1)
print(df_h4_periode[df_h4_periode["state_changed"]][["Close", "market_state"]])
df_h4_periode2 = df_result.loc['2026-06-15 00:00' : '2026-06-17 0:00']
print(df_h4_periode2)

                       Close market_state
Date                                     
2026-06-10 00:00:00  214.552      BEARISH
2026-06-10 08:00:00  214.868      NEUTRAL
2026-06-10 12:00:00  215.068      BULLISH
2026-06-11 12:00:00  213.914      NEUTRAL
2026-06-15 00:00:00  215.213      BULLISH
2026-06-16 00:00:00  214.649      NEUTRAL
2026-06-16 04:00:00  214.767      BEARISH
2026-06-16 12:00:00  215.372      NEUTRAL
2026-06-16 16:00:00  215.432      BULLISH
                        Open     High      Low    Close  Tickvol  \
Date                                                               
2026-06-15 00:00:00  215.143  215.356  215.035  215.213    26924   
2026-06-15 04:00:00  215.211  215.369  215.149  215.238    18316   
2026-06-15 08:00:00  215.243  215.281  214.884  214.961    24065   
2026-06-15 12:00:00  214.961  215.243  214.905  215.221    34753   
2026-06-15 16:00:00  215.221  215.241  215.023  215.117    23156   
2026-06-15 20:00:00  215.114  215.127  214.840  215.010    198

In [15]:
# process m15 data
df_m15 = df_m15_raw.loc[start_date : end_date]

m15_tf_direction_list = []
m15_tf_is_rangebox_list = []
m15_tf_er15_list = []

for i in range(len(df_result)):
    row = df_result.iloc[i]
    start_date_weekly = row.name
    end_date_weekly = start_date_weekly + timedelta(hours=3, minutes=59)
    df_by_day = df_m15.loc[f'{start_date_weekly}' : f'{end_date_weekly}']

    for index2, row2 in df_by_day.iterrows():
        m15_tf_direction_list.append(df_result.iloc[i-1]['market_state'])
        m15_tf_is_rangebox_list.append(df_result.iloc[i-1]['is_range_box'])
        m15_tf_er15_list.append(df_result.iloc[i-1]['ER'])

df_m15['market_state_h4'] = m15_tf_direction_list
df_m15['is_range_box'] = m15_tf_is_rangebox_list
df_m15['ER'] = m15_tf_er15_list

indicator_atr_14 = AverageTrueRange(df_m15['High'], df_m15['Low'], df_m15['Close'], window=14)
df_m15['ATR'] = indicator_atr_14._atr

indicator_rsi_14 = RSIIndicator(df_m15['Close'], window=14)
df_m15['RSI14'] = indicator_rsi_14._rsi
df_m15.index = pd.to_datetime(df_m15.index, utc=True).tz_convert(timezone)

In [ ]:
# process m15 data + m15 market structure
df_m15 = df_m15_raw.loc[start_date : end_date]
df_m15 = implement_market_structure_states_strict(df_m15, n=window)


df_m15_periode = df_m15.loc['2026-06-5 10:00' : '2026-06-5 19:00']
# Affichage des transitions d'états
df_m15_periode["state_changed"] = df_m15_periode["market_state"] != df_m15_periode[
    "market_state"
].shift(1)
print(df_m15_periode[df_m15_periode["state_changed"]][["Close", "market_state"]])
# df_m15_periode2 = df_m15.loc['2026-06-5 13:00' : '2026-06-5 14:00']
# print(df_m15_periode2)
# for row in df_m15_periode[df_m15_periode["state_changed"]][["Close", "market_state"]].iterrows():
#     print(row)


m15_tf_direction_list = []
m15_tf_is_rangebox_list = []
m15_tf_er15_list = []

for i in range(len(df_result)):
    row = df_result.iloc[i]
    start_date_weekly = row.name
    end_date_weekly = start_date_weekly + timedelta(hours=3, minutes=59)
    df_by_day = df_m15.loc[f'{start_date_weekly}' : f'{end_date_weekly}']

    for index2, row2 in df_by_day.iterrows():
        m15_tf_direction_list.append(df_result.iloc[i-1]['market_state'])
        m15_tf_is_rangebox_list.append(df_result.iloc[i-1]['is_range_box'])
        m15_tf_er15_list.append(df_result.iloc[i-1]['ER'])

df_m15['market_state_h4'] = m15_tf_direction_list
df_m15['is_range_box'] = m15_tf_is_rangebox_list
df_m15['ER'] = m15_tf_er15_list

indicator_atr_14 = AverageTrueRange(df_m15['High'], df_m15['Low'], df_m15['Close'], window=14)
df_m15['ATR'] = indicator_atr_14._atr

indicator_rsi_14 = RSIIndicator(df_m15['Close'], window=14)
df_m15['RSI14'] = indicator_rsi_14._rsi

print(df_m15)

                       Close market_state
Date                                     
2026-06-05 10:00:00  215.330      BULLISH
2026-06-05 14:00:00  214.649      NEUTRAL
2026-06-05 16:15:00  214.351      BEARISH
                        Open     High      Low    Close  Tickvol  \
Date                                                               
2022-01-03 00:00:00  155.676  155.776  155.676  155.731     1032   
2022-01-03 00:15:00  155.731  155.754  155.707  155.715      412   
2022-01-03 00:30:00  155.715  155.731  155.672  155.680      577   
2022-01-03 00:45:00  155.680  155.704  155.669  155.688      351   
2022-01-03 01:00:00  155.684  155.710  155.664  155.674      517   
...                      ...      ...      ...      ...      ...   
2026-09-15 23:00:00  208.895  209.029  208.895  209.018     1853   
2026-09-15 23:15:00  209.020  209.027  208.982  208.996     1268   
2026-09-15 23:30:00  208.997  209.013  208.926  208.953     1279   
2026-09-15 23:45:00  208.951  209.021  208

In [66]:
# run backtest
bt = Backtest(df_m15, OpeningRangeBreakout, margin=0.02, finalize_trades=True)

stats = bt.run()

print(stats)

bt.plot()

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2022-01-19 16:30:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.com/kernc/backtesting.py/issues/119
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2022-02-23 16:00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.c

Start                     2022-01-03 00:00:00
End                       2026-09-16 00:00:00
Duration                   1717 days 00:00:00
Exposure Time [%]                     0.88266
Equity Final [$]                   9850.65152
Equity Peak [$]                   10740.92804
Return [%]                           -1.49348
Buy & Hold Return [%]                34.25394
Return (Ann.) [%]                    -0.30857
Volatility (Ann.) [%]                 7.11604
CAGR [%]                             -0.31959
Sharpe Ratio                         -0.04336
Sortino Ratio                        -0.06147
Calmar Ratio                         -0.01814
Alpha [%]                            -1.92542
Beta                                  0.01261
Max. Drawdown [%]                   -17.01194
Avg. Drawdown [%]                    -2.28906
Max. Drawdown Duration     1391 days 10:30:00
Avg. Drawdown Duration      109 days 09:14:00
# Trades                                  220
Win Rate [%]                      

GridPlot(id='p4783', ...)